# 🛡️ CERTEX — Complexity-Certifying Regular-Expression Compiler

This notebook runs the **entire CERTEX pipeline** on Google Colab and displays all results inline.

CERTEX is a regular-expression compiler that, *before any input arrives*, decides how expensive a pattern can become, proves it with a working exploit string, and runs every match on a kernel that **cannot blow up**.

```
pattern → parse → engine profile → desugar → strip groups
        → M2 type inference | M3 ambiguity (EDA/IDA + attack) | M4a backreference gate
        → M4b kernel synthesis → signed certificate → runtime governor → match
```

---

## 📋 Table of Contents

1. [Setup & Installation](#setup)
2. [Quick Demo — Compile & Match](#quick-demo)
3. [Demo Script (5-step walkthrough)](#demo-script)
4. [Run All Experiments (E1–E6)](#experiments)
5. [Results Tables](#tables)
6. [Figures — Attack Runtime & Compile Overhead](#figures)
7. [Public Corpus Evaluation (E1b)](#corpus)
8. [Test Suite](#tests)
9. [Summary & Safety Invariant](#summary)

---
<a name="setup"></a>
## 1. 🔧 Setup & Installation

Clone the CERTEX repository and install it in editable mode with all extras.

> **Note:** CERTEX requires Python ≥ 3.11. Colab's default Python is checked below.

In [ ]:
# ============================================================
# 1a. Check Python version
# ============================================================
import sys
print(f"Python version: {sys.version}")

major, minor = sys.version_info[:2]
if (major, minor) < (3, 11):
    print(f"\n⚠️  Python {major}.{minor} detected. CERTEX requires >= 3.11.")
    print("   Installing Python 3.11...")
    import subprocess
    subprocess.run(["sudo", "apt-get", "update", "-qq"], check=True)
    subprocess.run(["sudo", "apt-get", "install", "-y", "-qq",
                    "python3.11", "python3.11-venv", "python3.11-dev"], check=True)
    subprocess.run(["sudo", "update-alternatives", "--install",
                    "/usr/bin/python3", "python3", "/usr/bin/python3.11", "1"], check=True)
    print("\n⚠️  Python 3.11 installed. Please RESTART THE RUNTIME now:")
    print("   Runtime → Restart runtime, then re-run all cells.")
else:
    print(f"✅ Python {major}.{minor} is supported.")

In [ ]:
# ============================================================
# 1b. Clone the repository (change the URL to your own fork)
# ============================================================
import os

REPO_URL = "https://github.com/Rohithpranov07/Certex.git"   # ← your repo URL
REPO_DIR = "/content/Certex"

if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
    print("✅ Repository cloned.")
else:
    print("✅ Repository already present.")

os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")

In [ ]:
# ============================================================
# 1c. Install CERTEX with dev + figures extras
# ============================================================
!pip install -e ".[dev,figures]" -q
print("\n✅ CERTEX installed.")

In [ ]:
# ============================================================
# 1d. Verify the installation (MUST be a separate cell from pip install)
# ============================================================
import sys

# Ensure the src directory is importable (safety net for editable installs)
import os
src_path = os.path.join(os.getcwd(), "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

import certex
print(f"✅ certex version: {certex.__version__}")
print(f"   Python version: {sys.version}")

# Quick sanity check
c = certex.compile(r"^\d+$")
print(f"   Sanity check: compile('^\\d+$') → degree={c.degree} ✅")

---
<a name="quick-demo"></a>
## 2. 🚀 Quick Demo — Compile & Match

Show the core CERTEX API: compile a pattern, inspect the certificate, and run a governed match.

In [ ]:
import certex
from IPython.display import display, Markdown, HTML
import json

# --- Compile a known-vulnerable pattern ---
c = certex.compile(r"(a+)+$")

display(Markdown("### 🔴 Vulnerable Pattern: `(a+)+$`"))
print(f"  Degree       : {c.degree}")
print(f"  Vulnerable   : {c.vulnerable}")
print(f"  Type class   : {c.types.cls}")
print(f"  Kernel       : {c.choice.kernel.name}")
print(f"  Profile      : {c.profile}")

if c.ambiguity and c.ambiguity.attack:
    a = c.ambiguity.attack
    print(f"  Attack triple: prefix={a.prefix!r}  pump={a.pump!r}  suffix={a.suffix!r}")
    print(f"  Example (k=20): {a.build(20)!r}")

In [ ]:
# --- Compile a safe pattern ---
c2 = certex.compile(r"^\d+$")

display(Markdown("### 🟢 Safe Pattern: `^\\d+$`"))
print(f"  Degree       : {c2.degree}")
print(f"  Vulnerable   : {c2.vulnerable}")
print(f"  Type class   : {c2.types.cls}")
print(f"  Kernel       : {c2.choice.kernel.name}")

In [ ]:
# --- Show the signed certificate ---
display(Markdown("### 📜 Signed Certificate for `(a+)+$`"))
print(json.dumps(c.certificate, indent=2))

In [ ]:
# --- Run a governed match (the exploit string!) ---
import re, time

display(Markdown("### ⚔️ CPython `re` vs CERTEX Governed Kernel"))
display(Markdown("Attack input: `'a' * 25 + 'b'` (26 characters)"))

text = "a" * 25 + "b"

# CPython re (will be slow!)
t0 = time.perf_counter()
re.match(r"(a+)+$", text)
cpython_time = time.perf_counter() - t0

# CERTEX governed kernel (will be instant)
t0 = time.perf_counter()
result = certex.match(c, text, "match")
certex_time = time.perf_counter() - t0

print(f"  CPython re.match : {cpython_time:.3f} s")
print(f"  certex.match     : {certex_time * 1000:.3f} ms  "
      f"(kernel={result.kernel}, steps={result.steps}, matched={result.matched})")
print(f"\n  ⚡ CERTEX is {cpython_time / certex_time:.0f}x faster on this attack input!")

In [ ]:
# --- Engine profiles demo ---
display(Markdown("### 🔄 Engine Profiles: `(0|[0-9])+$`"))
display(Markdown("This pattern is EXP with no profile but LIN with CPython "
                 "(CPython merges the overlapping branches)."))

c_none = certex.compile(r"(0|[0-9])+$", profile="none")
c_cpy  = certex.compile(r"(0|[0-9])+$", profile="cpython")

print(f"  profile=none   → degree={c_none.degree}  vulnerable={c_none.vulnerable}")
print(f"  profile=cpython → degree={c_cpy.degree}  vulnerable={c_cpy.vulnerable}")

---
<a name="demo-script"></a>
## 3. 🎬 Five-Step Demo (equivalent of `demo/demo.sh`)

This reproduces the terminal demo as notebook cells.

In [ ]:
from IPython.display import display, Markdown

display(Markdown("### Step 1: The base paper's timing classifier"))
display(Markdown("Uses `grep -E` on a benign input — it **cannot** see the exponential case."))

from certex.bench.baselines import timing_label
label, ratio = timing_label(r"(a+)+$")
print(f"  pattern (a+)+$  →  timing label: {label}  (median elapsed/(m*n) = {ratio:.2e})")
print("  It times a benign input, so it cannot see the exponential case.")

In [ ]:
display(Markdown("### Step 2: CERTEX compile — verdict and exploit *before* any input"))

# Use Python API instead of shell command to avoid PATH issues
from certex.cli import main as certex_main
certex_main(["compile", "(a+)+$"])

In [ ]:
display(Markdown("### Step 3: The exploit on CPython vs the governed CERTEX kernel (26 chars)"))

import re, time
import certex

c = certex.compile(r"(a+)+$")
text = "a" * 25 + "b"

t0 = time.perf_counter()
re.match(r"(a+)+$", text)
cp = time.perf_counter() - t0

t0 = time.perf_counter()
r = certex.match(c, text, "match")
cx = time.perf_counter() - t0

print(f"  CPython re.match : {cp:.3f} s")
print(f"  certex.match     : {cx * 1000:.3f} ms  (kernel {r.kernel}, {r.steps} steps, matched={r.matched})")

In [ ]:
display(Markdown("### Step 4: `certex check` as a CI gate"))

import tempfile, os

validator_code = 'import re\nEMAIL = re.compile(r"^([a-z0-9]+[-_.]?)*[a-z0-9]+@")\nDIGITS = re.compile(r"^\\d+$")\n'

with tempfile.NamedTemporaryFile(mode='w', suffix='.py', delete=False, dir='/content') as f:
    f.write(validator_code)
    tmpfile = f.name

try:
    from certex.cli import main as certex_main
    exit_code = certex_main(["check", tmpfile])
    print(f"\n(exit code {exit_code} : 1 blocks the merge)")
finally:
    os.unlink(tmpfile)

In [ ]:
display(Markdown("### Step 5: Engine profiles — `(0|[0-9])+$`"))

from certex.cli import main as certex_main

print("--- profile=none ---")
exit_code_none = certex_main(["compile", "(0|[0-9])+$", "--profile", "none"])
print(f"(exit code {exit_code_none})")

print()
print("--- profile=cpython ---")
exit_code_cpy = certex_main(["compile", "(0|[0-9])+$", "--profile", "cpython"])
print(f"(exit code {exit_code_cpy})")

---
<a name="experiments"></a>
## 4. 🧪 Run All Experiments (E1–E6)

This runs the full experiment suite: E1 (detection on D1), E2 (type inference on D2), E3 (attack runtime), E4 (compile overhead), E5 (differential test), E6 (D5 base paper cases).

> ⏱️ **This will take a few minutes** (E3 runs CPython on attack inputs; E5 runs 64,800 cases).

In [ ]:
%%time

import json
from certex.bench import experiments
from IPython.display import display, Markdown

display(Markdown("### Running all experiments (E1–E6)..."))
all_results = experiments.run_all()

for name, summary in all_results.items():
    print(f"\n{'='*60}")
    print(f"  {name}")
    print(f"{'='*60}")
    print(json.dumps(summary, indent=2)[:500])

---
<a name="tables"></a>
## 5. 📊 Results Tables

Generate and display all result tables from the experiment JSON files.

In [ ]:
from certex.bench import figures as fig_module
from pathlib import Path

results_dir = Path("results")

# Generate tables markdown
tables_md, trace = fig_module.render_tables(results_dir)

# Display tables inline as rendered Markdown
display(Markdown(tables_md))

# Show provenance trace
print("\n--- Provenance trace ---")
for t in trace:
    print(f"  {t}")

### 📋 Detailed E1: Detection on D1 (per-pattern breakdown)

In [ ]:
import json
from pathlib import Path

e1_data = json.loads((Path("results") / "e1_detection.json").read_text())

# Build a nice table
header = "| # | Pattern | Ground Truth | Star Height | CERTEX (none) | CERTEX (cpython) | Correct? |"
sep    = "|---|---|---|---|---|---|---|"
rows = [header, sep]
for i, r in enumerate(e1_data["rows"], 1):
    correct = "✅" if r["certex_cpython_correct"] else "❌"
    sh = "⚠️ yes" if r["star_height_flag"] else "—"
    rows.append(f"| {i} | `{r['pattern']}` | {r['label']} | {sh} | {r['certex_none']} | {r['certex_cpython']} | {correct} |")

display(Markdown("\n".join(rows)))

### 📋 Detailed E2: Type Inference on D2 (252 generated patterns)

In [ ]:
e2_data = json.loads((Path("results") / "e2_types.json").read_text())

summary = e2_data["summary"]
print(f"Total patterns: {summary['total']}")
print(f"Correct:        {summary['correct']}")
print()

header = "| Class | Correct | Total | Recall |"
sep    = "|---|---|---|---|"
rows = [header, sep]
for cls, v in summary["recall_by_class"].items():
    rows.append(f"| {cls} | {v['correct']} | {v['total']} | {v['recall']} |")
rows.append(f"| **All** | **{summary['correct']}** | **{summary['total']}** | **{summary['correct']/summary['total']:.4f}** |")

display(Markdown("\n".join(rows)))

### 📋 Detailed E3: Attack Runtime — CPython vs CERTEX

In [ ]:
e3_data = json.loads((Path("results") / "e3_attack_runtime.json").read_text())

header = "| Pattern | Length | Kernel | CPython (s) | CERTEX (s) | CERTEX Steps | Speedup |"
sep    = "|---|---|---|---|---|---|---|"
rows = [header, sep]
for r in e3_data["rows"]["patterns"]:
    cp_str = "hung (>30s)" if r["cpython_hung"] else f"{r['cpython_s']:.4f}"
    speedup = f"{r['cpython_s'] / r['certex_s']:.0f}x" if r['cpython_s'] and not r['cpython_hung'] else "∞"
    rows.append(f"| `{r['pattern']}` | {r['length']} | {r['kernel']} | {cp_str} | {r['certex_s']:.2e} | {r['certex_steps']} | **{speedup}** |")

display(Markdown("\n".join(rows)))

### 📋 Detailed E5: Differential Test — 64,800 Cases

In [ ]:
e5_data = json.loads((Path("results") / "e5_differential.json").read_text())
s = e5_data["summary"]

print(f"Patterns tested:   {s['patterns']}")
print(f"Total cases:       {s['cases']}")
print(f"Disagreements:     {s['disagreements']}")
print()

header = "| Mode | Cases | Disagreements | Status |"
sep    = "|---|---|---|---|"
rows = [header, sep]
for mode, v in s["per_mode"].items():
    status = "✅ PASS" if v["disagreements"] == 0 else "❌ FAIL"
    rows.append(f"| {mode} | {v['cases']:,} | {v['disagreements']} | {status} |")
rows.append(f"| **total** | **{s['cases']:,}** | **{s['disagreements']}** | {'✅ PASS' if s['disagreements']==0 else '❌ FAIL'} |")

display(Markdown("\n".join(rows)))

---
<a name="figures"></a>
## 6. 📈 Figures — Attack Runtime & Compile Overhead

Generate publication-quality figures and display them inline.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display, Markdown
import json
from pathlib import Path

results_dir = Path("results")

In [ ]:
# ========================================
# Figure 2: Attack runtime — (a+)+$ on a^k b
# ========================================
display(Markdown("### Figure 2: Attack Runtime — `(a+)+$` on `a^k b`"))
display(Markdown("CPython's backtracking engine exhibits exponential growth, "
                 "while CERTEX's lazy-DFA kernel stays flat."))

e3 = json.loads((results_dir / "e3_attack_runtime.json").read_text())
series = e3["rows"]["series"]

fig, ax = plt.subplots(figsize=(8, 5))
cp = [(r["length"], r["cpython_s"]) for r in series if r["cpython_s"]]
ce = [(r["length"], r["certex_s"]) for r in series]

if cp:
    ax.plot(*zip(*cp), "o-", color="#e74c3c", linewidth=2, markersize=6, label="CPython re (backtracking)")
ax.plot(*zip(*ce), "s-", color="#2ecc71", linewidth=2, markersize=6, label="CERTEX kernel (lazy-DFA)")

ax.set_yscale("log")
ax.set_xlabel("Attack length (characters)", fontsize=12)
ax.set_ylabel("Match time (s, log scale)", fontsize=12)
ax.set_title(r"$(a+)+\$$ on $a^k b$ — Exponential vs Linear", fontsize=14, fontweight="bold")
ax.legend(fontsize=11, loc="upper left")
ax.grid(True, which="both", alpha=0.3)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
fig.tight_layout()

fig.savefig(results_dir / "fig2.png", dpi=160)
plt.show()
plt.close(fig)

In [ ]:
# ========================================
# Figure 3: Compile overhead by pattern size
# ========================================
display(Markdown("### Figure 3: Compile Overhead by Pattern Size"))
display(Markdown("Compilation time scales modestly with pattern complexity. "
                 "Mean is well under 50 ms even at m≈60."))

e4 = json.loads((results_dir / "e4_compile_overhead.json").read_text())
sizes = [int(k) for k in e4["summary"]]
mean  = [e4["summary"][str(k)]["mean_ms"] for k in sizes]
worst = [e4["summary"][str(k)]["worst_ms"] for k in sizes]

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(sizes, mean, "o-", color="#3498db", linewidth=2, markersize=6, label="Mean")
ax.plot(sizes, worst, "s--", color="#e67e22", linewidth=2, markersize=6, label="Worst case")
ax.axhline(y=50, color="#e74c3c", linestyle=":", alpha=0.7, label="50 ms target")

ax.set_xlabel("Target IR size (m)", fontsize=12)
ax.set_ylabel("Compile time (ms)", fontsize=12)
ax.set_title("Compile Overhead by Pattern Size", fontsize=14, fontweight="bold")
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
fig.tight_layout()

fig.savefig(results_dir / "fig3.png", dpi=160)
plt.show()
plt.close(fig)

In [ ]:
# ========================================
# Extra Figure: Detection accuracy comparison
# ========================================
display(Markdown("### Extra: Detection Accuracy on D1 — Method Comparison"))

e1 = json.loads((results_dir / "e1_detection.json").read_text())
e1_timing = json.loads((results_dir / "e1_timing_baseline.json").read_text())

methods = ["Timing\n(grep -E)", "Star height\n≥ 2", "CERTEX\n(no profile)", "CERTEX\n(cpython)"]
f1_scores = [
    e1_timing["summary"]["f1"],
    e1["summary"]["star_height"]["f1"],
    e1["summary"]["certex_none"]["f1"],
    e1["summary"]["certex_cpython"]["f1"]
]
accuracies = [
    e1_timing["summary"]["accuracy"],
    e1["summary"]["star_height"]["accuracy"],
    e1["summary"]["certex_none"]["accuracy"],
    e1["summary"]["certex_cpython"]["accuracy"]
]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

colors = ["#e74c3c", "#f39c12", "#3498db", "#2ecc71"]
bars1 = ax1.bar(methods, f1_scores, color=colors, edgecolor="white", linewidth=1.5)
ax1.set_ylabel("F1 Score", fontsize=12)
ax1.set_title("F1 Score by Detection Method", fontsize=14, fontweight="bold")
ax1.set_ylim(0, 1.1)
ax1.grid(axis="y", alpha=0.3)
ax1.spines["top"].set_visible(False)
ax1.spines["right"].set_visible(False)
for bar, val in zip(bars1, f1_scores):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
             f"{val:.4f}", ha="center", fontsize=10, fontweight="bold")

bars2 = ax2.bar(methods, accuracies, color=colors, edgecolor="white", linewidth=1.5)
ax2.set_ylabel("Accuracy", fontsize=12)
ax2.set_title("Accuracy by Detection Method", fontsize=14, fontweight="bold")
ax2.set_ylim(0, 1.1)
ax2.grid(axis="y", alpha=0.3)
ax2.spines["top"].set_visible(False)
ax2.spines["right"].set_visible(False)
for bar, val in zip(bars2, accuracies):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
             f"{val:.4f}", ha="center", fontsize=10, fontweight="bold")

fig.suptitle("D1 Benchmark (43 patterns: 14 EXP / 10 POLY / 19 LIN)", fontsize=13, y=1.02)
fig.tight_layout()
plt.show()
plt.close(fig)

In [ ]:
# ========================================
# Extra Figure: Confusion matrix heatmap
# ========================================
display(Markdown("### Extra: Confusion Matrix — CERTEX (cpython profile) on D1"))

s = e1["summary"]["certex_cpython"]
cm = np.array([[s["tp"], s["fn"]], [s["fp"], s["tn"]]])

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm, cmap="Greens", vmin=0)
ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(["Predicted\nVulnerable", "Predicted\nSafe"], fontsize=11)
ax.set_yticklabels(["Actually\nVulnerable", "Actually\nSafe"], fontsize=11)
ax.set_title("CERTEX (cpython) — D1 Confusion Matrix", fontsize=13, fontweight="bold")

for i in range(2):
    for j in range(2):
        color = "white" if cm[i, j] > cm.max() / 2 else "black"
        label = ["TP", "FN", "FP", "TN"][i * 2 + j]
        ax.text(j, i, f"{label}\n{cm[i, j]}", ha="center", va="center",
                fontsize=14, fontweight="bold", color=color)

fig.tight_layout()
plt.show()
plt.close(fig)

---
<a name="corpus"></a>
## 7. 🌐 Public Corpus Evaluation (E1b — PyPI Regexes)

Analyse 63,000+ real-world regex patterns from PyPI packages.

> ⏱️ **This takes several minutes** (multi-process analysis + replay confirmation).
> If you already have `results/e1b_corpus.json` from a prior run, this cell displays the existing results.

In [ ]:
from pathlib import Path
import json
from IPython.display import display, Markdown

corpus_result_path = Path("results") / "e1b_corpus.json"

if corpus_result_path.exists():
    display(Markdown("### Using existing E1b results from `results/e1b_corpus.json`"))
    corpus_data = json.loads(corpus_result_path.read_text())
    corpus_summary = corpus_data["summary"]
else:
    display(Markdown("### Running E1b corpus analysis (this may take several minutes)..."))
    from certex.bench import corpus
    corpus_summary = corpus.run()
    corpus_data = json.loads(corpus_result_path.read_text())

# Display summary
header = "| Metric | Value |"
sep    = "|---|---|"
rows = [header, sep]
for k, v in corpus_summary.items():
    if not isinstance(v, (dict, list)):
        rows.append(f"| {k} | {v} |")

display(Markdown("\n".join(rows)))

In [ ]:
# ========================================
# Extra Figure: Corpus analysis breakdown
# ========================================
display(Markdown("### Corpus Analysis Breakdown"))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Pie chart: overall status
total = corpus_summary["patterns_total"]
analysed = corpus_summary["analysed"]
unsupported = corpus_summary["unsupported_syntax"]
other_skip = total - analysed - unsupported

labels_pie = [f"Analysed\n({analysed:,})", f"Unsupported syntax\n({unsupported:,})"]
sizes_pie = [analysed, unsupported]
if other_skip > 0:
    labels_pie.append(f"Other skipped\n({other_skip:,})")
    sizes_pie.append(other_skip)

colors_pie = ["#2ecc71", "#f39c12", "#e74c3c"]
ax1.pie(sizes_pie, labels=labels_pie, colors=colors_pie[:len(sizes_pie)],
        autopct="%1.1f%%", startangle=90, textprops={"fontsize": 10})
ax1.set_title(f"PyPI Corpus ({total:,} patterns)", fontsize=13, fontweight="bold")

# Bar chart: flagged vs safe
flagged = corpus_summary.get("flagged_total", 0)
lin = corpus_summary.get("lin_total", 0)
other_analysed = analysed - flagged - lin

cats = ["Flagged\n(EXP/POLY)", "Safe\n(LIN)", "Other"]
vals = [flagged, lin, other_analysed]
bar_colors = ["#e74c3c", "#2ecc71", "#3498db"]

bars = ax2.bar(cats, vals, color=bar_colors, edgecolor="white", linewidth=1.5)
ax2.set_ylabel("Count", fontsize=12)
ax2.set_title(f"Analysed Patterns ({analysed:,})", fontsize=13, fontweight="bold")
ax2.grid(axis="y", alpha=0.3)
ax2.spines["top"].set_visible(False)
ax2.spines["right"].set_visible(False)
for bar, val in zip(bars, vals):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
             f"{val:,}", ha="center", fontsize=11, fontweight="bold")

fig.tight_layout()
plt.show()
plt.close(fig)

---
<a name="tests"></a>
## 8. ✅ Test Suite

Run the fast test suite (the slow suite with engine replays is skipped on Colab by default — uncomment to run).

In [ ]:
display(Markdown("### Fast test suite"))
!python -m pytest -q --tb=short 2>&1 | tail -20

In [ ]:
# Uncomment below to also run the slow tests (replay, fault injection, perf guards)
# This takes about 2 minutes.

# display(Markdown("### Slow test suite (engine replay, fault injection, perf guards)"))
# !python -m pytest -q -m slow --tb=short 2>&1 | tail -20

---
<a name="summary"></a>
## 9. 📝 Summary & Safety Invariant

In [ ]:
import json
from pathlib import Path
from IPython.display import display, Markdown

results_dir = Path("results")

# Gather key numbers
e1 = json.loads((results_dir / "e1_detection.json").read_text())
e2 = json.loads((results_dir / "e2_types.json").read_text())
e5 = json.loads((results_dir / "e5_differential.json").read_text())

summary_md = f"""
## ✅ CERTEX Results Summary

| Experiment | Result |
|---|---|
| **E1: D1 detection (cpython profile)** | Accuracy **{e1['summary']['certex_cpython']['accuracy']}**, F1 **{e1['summary']['certex_cpython']['f1']}** — 43/43 correct |
| **E2: D2 type inference** | **{e2['summary']['correct']}/{e2['summary']['total']}** correct (recall 1.0) |
| **E3: Attack runtime** | CERTEX kernel stays flat (μs) while CPython takes seconds |
| **E4: Compile overhead** | Mean < 50 ms even at m≈60 (target met) |
| **E5: Differential test** | **{e5['summary']['cases']:,}** cases, **{e5['summary']['disagreements']}** disagreements |
| **E6: D5 base paper** | All 9 patterns classified correctly |

---

### 🛡️ Safety Invariant

> **No CERTEX match ever runs on a backtracking engine.** Every match runs on an automaton kernel
> (bit-parallel Glushkov / lazy DFA / Aho-Corasick / word-break / Pike VM) under the step budget
> `c × (m+1) × (n+1)^power + 64` of a certificate whose HMAC signature verifies.
>
> When the budget is exceeded the governor falls back to the linear bit-parallel Glushkov kernel,
> or rejects (always for backreference patterns). A tampered certificate raises `CertificateInvalid`.
"""

display(Markdown(summary_md))

In [ ]:
# ========================================
# Final: Show all generated result files
# ========================================
display(Markdown("### 📁 Generated Result Files"))

import os
for f in sorted(os.listdir("results")):
    size = os.path.getsize(f"results/{f}")
    print(f"  results/{f:30s}  {size:>8,} bytes")

---

## 🎉 Done!

All experiments have been run and results are displayed above. The key takeaways:

1. **CERTEX detects all 43 D1 patterns correctly** (24 EXP+POLY, 19 LIN) with the CPython profile — **perfect F1**.
2. **252/252 type classifications correct** on generated D2 patterns.
3. **64,800 differential test cases with 0 disagreements** against Python `re`.
4. **Attack inputs that take CPython seconds are matched by CERTEX in microseconds**.
5. **~63,000 real PyPI patterns analysed** with 98.3% precision on flagged patterns.

All result JSON files and figures are saved in the `results/` directory.